In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt

In [3]:
state_dict = torch.load("PatchEmbed_Q7.16.pt", map_location="cpu")

print("Total layers:", len(state_dict))

Total layers: 25


In [4]:
for k in state_dict.keys():
    print(k)

cls_token
pos_embed
cnn.block1.0.weight
cnn.block2.0.weight
cnn.block3.0.weight
patch_embed.proj.weight
patch_embed.proj.bias
patch_embed.norm.weight
patch_embed.norm.bias
blocks.0.norm1.weight
blocks.0.norm1.bias
blocks.0.attn.qkv.weight
blocks.0.attn.qkv.bias
blocks.0.attn.proj.weight
blocks.0.attn.proj.bias
blocks.0.norm2.weight
blocks.0.norm2.bias
blocks.0.mlp.fc1.weight
blocks.0.mlp.fc1.bias
blocks.0.mlp.fc2.weight
blocks.0.mlp.fc2.bias
norm.weight
norm.bias
head.weight
head.bias


In [5]:
for name, param in state_dict.items():
    w = param.numpy()

    print(f"{name:30s} "
          f"shape={w.shape} "
          f"min={w.min():.6f} "
          f"max={w.max():.6f} "
          )

cls_token                      shape=(1, 1, 32) min=-0.259401 max=0.341372 
pos_embed                      shape=(1, 26, 32) min=-1.118263 max=1.123286 
cnn.block1.0.weight            shape=(4, 1, 3, 3) min=-0.299896 max=0.554855 
cnn.block2.0.weight            shape=(4, 4, 3, 3) min=-0.727585 max=0.447281 
cnn.block3.0.weight            shape=(12, 4, 3, 3) min=-0.796555 max=0.469604 
patch_embed.proj.weight        shape=(32, 12, 2, 2) min=-0.776413 max=0.632889 
patch_embed.proj.bias          shape=(32,) min=-0.418854 max=0.267944 
patch_embed.norm.weight        shape=(32,) min=0.579592 max=1.059300 
patch_embed.norm.bias          shape=(32,) min=-0.521249 max=0.571801 
blocks.0.norm1.weight          shape=(32,) min=0.765574 max=1.230283 
blocks.0.norm1.bias            shape=(32,) min=-0.486769 max=0.494289 
blocks.0.attn.qkv.weight       shape=(96, 32) min=-0.653071 max=0.695120 
blocks.0.attn.qkv.bias         shape=(96,) min=-0.288378 max=0.251554 
blocks.0.attn.proj.weight      sha

In [6]:
for name, param in state_dict.items():
    if "patch_embed" in name.lower():   # 只抓 patch_embed
        w = param.numpy()

        print(f"{name:30s} "
              f"shape={w.shape} "
              f"min={w.min():.6f} "
              f"max={w.max():.6f}")

patch_embed.proj.weight        shape=(32, 12, 2, 2) min=-0.776413 max=0.632889
patch_embed.proj.bias          shape=(32,) min=-0.418854 max=0.267944
patch_embed.norm.weight        shape=(32,) min=0.579592 max=1.059300
patch_embed.norm.bias          shape=(32,) min=-0.521249 max=0.571801


In [8]:
frac_bits = 16
scale = 2 ** frac_bits

for name, param in state_dict.items():
    if "patch_embed" in name.lower():
        w = param.numpy()

        w_min = w.min()
        w_max = w.max()

        # 對應硬體整數值
        int_min = int(round(w_min * scale))
        int_max = int(round(w_max * scale))

        print(f"{name:30s} "
              f"min={w_min:+.25f} "
              f"max={w_max:+.25f} "
              f"| int_min={int_min:+8d} "
              f"int_max={int_max:+8d}")

patch_embed.proj.weight        min=-0.7764129638671875000000000 max=+0.6328887939453125000000000 | int_min=  -50883 int_max=  +41477
patch_embed.proj.bias          min=-0.4188537597656250000000000 max=+0.2679443359375000000000000 | int_min=  -27450 int_max=  +17560
patch_embed.norm.weight        min=+0.5795924663543701171875000 max=+1.0592995882034301757812500 | int_min=  +37984 int_max=  +69422
patch_embed.norm.bias          min=-0.5212492346763610839843750 max=+0.5718007087707519531250000 | int_min=  -34161 int_max=  +37474
